# Notebook 5 — Model Evaluation

Creates model-comparison tables, ROC and precision-recall curves, confusion matrices, learning curves, and Precision@5 results for every completed experiment.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, roc_auc_score, precision_recall_curve, average_precision_score

In [2]:
MODEL_ROOT=Path("../models")
FEATURE_ROOT=Path("../data/features")
RESULT_ROOT=Path("../results")

In [3]:
RUN_EXPERIMENTS = [
    "monthly_ff3",
    "monthly_ff5",
    "daily_ff3",
    "daily_ff5",
]

EXPERIMENT_CONFIG = {
    "monthly_ff3": {"frequency": "monthly", "factor_model": "ff3", "window": 12, "periods_per_year": 12},
    "monthly_ff5": {"frequency": "monthly", "factor_model": "ff5", "window": 12, "periods_per_year": 12},
    "daily_ff3": {"frequency": "daily", "factor_model": "ff3", "window": 60, "periods_per_year": 252},
    "daily_ff5": {"frequency": "daily", "factor_model": "ff5", "window": 60, "periods_per_year": 252},
}

unknown = set(RUN_EXPERIMENTS) - set(EXPERIMENT_CONFIG)
if unknown:
    raise ValueError(f"Unknown experiments: {sorted(unknown)}")

In [4]:
PROBABILITY_COLUMNS={
    "XGBoost Signature":"xgb_signature_prob","XGBoost GARCH":"xgb_garch_prob","XGBoost Combined":"xgb_combined_prob",
    "CNN Signature":"cnn_signature_prob","CNN GARCH":"cnn_garch_prob","CNN Combined":"cnn_combined_prob",
    "Ensemble Signature":"ensemble_signature_prob","Ensemble GARCH":"ensemble_garch_prob","Ensemble Combined":"ensemble_combined_prob",
}

def evaluate_top_k(data, probability_column, k=5):
    rows=[]
    for date,g in data.groupby("Date"):
        selected=g.nlargest(k,probability_column)
        correct=int(selected["actual_label"].sum())
        rows.append({"Date":date,"correct_selections":correct,"precision_at_k":correct/len(selected)})
    return pd.DataFrame(rows)

In [5]:
for experiment in RUN_EXPERIMENTS:
    model_dir=MODEL_ROOT/experiment; feature_dir=FEATURE_ROOT/experiment; out_dir=RESULT_ROOT/experiment
    out_dir.mkdir(parents=True,exist_ok=True)
    required=[model_dir/"final_model_results.csv",model_dir/"test_probabilities.csv",feature_dir/"meta_test.csv",feature_dir/"y_test.npy"]
    if not all(p.exists() for p in required):
        print("Skipping incomplete experiment:",experiment); continue
    results=pd.read_csv(model_dir/"final_model_results.csv")
    probs=pd.read_csv(model_dir/"test_probabilities.csv")
    meta=pd.read_csv(feature_dir/"meta_test.csv",parse_dates=["Date","RealizationDate"])
    y=np.load(feature_dir/"y_test.npy")
    assert len(probs)==len(meta)==len(y)
    results["model"]=results["model"].str.replace(" — Test","",regex=False)
    comparison=results.sort_values(["f1","roc_auc"],ascending=False).reset_index(drop=True)
    comparison.to_csv(out_dir/"model_comparison.csv",index=False)

    detail=meta.copy(); detail["actual_label"]=y
    for col in PROBABILITY_COLUMNS.values(): detail[col]=probs[col].to_numpy()
    detail.to_csv(out_dir/"prediction_detail.csv",index=False)

    top_rows=[]
    plt.figure(figsize=(10,7))
    for name,col in PROBABILITY_COLUMNS.items():
        fpr,tpr,_=roc_curve(y,probs[col]); auc=roc_auc_score(y,probs[col])
        plt.plot(fpr,tpr,label=f"{name} ({auc:.3f})")
        tk=evaluate_top_k(detail,col,5)
        top_rows.append({"model":name,"mean_precision_at_5":tk["precision_at_k"].mean(),"perfect_selection_rate":(tk["correct_selections"]==5).mean()})
    plt.plot([0,1],[0,1],"--"); plt.title(f"ROC Curves — {experiment}"); plt.legend(fontsize=8); plt.tight_layout(); plt.savefig(out_dir/"roc_curves.png",dpi=200); plt.close()

    plt.figure(figsize=(10,7))
    for name,col in PROBABILITY_COLUMNS.items():
        precision,recall,_=precision_recall_curve(y,probs[col]); ap=average_precision_score(y,probs[col])
        plt.plot(recall,precision,label=f"{name} ({ap:.3f})")
    plt.axhline(y.mean(),linestyle="--"); plt.title(f"Precision-Recall — {experiment}"); plt.legend(fontsize=8); plt.tight_layout(); plt.savefig(out_dir/"precision_recall.png",dpi=200); plt.close()

    pd.DataFrame(top_rows).sort_values("mean_precision_at_5",ascending=False).to_csv(out_dir/"top_k_summary.csv",index=False)
    comparison.assign(
        architecture=comparison["model"].str.extract(r"^(XGBoost|CNN|Ensemble)")[0],
        feature_type=comparison["model"].str.extract(r"(Signature|GARCH|Combined)$")[0]
    ).to_csv(out_dir/"model_comparison_enriched.csv",index=False)
    print("Evaluated",experiment)

Evaluated monthly_ff3
Evaluated monthly_ff5
Evaluated daily_ff3
Evaluated daily_ff5
